4. Build a small end-to-end ELT: read CSV, filter, add a column (e.g., ingestion_date), and write the
result as Delta.

In [0]:
%python
from pyspark.sql.functions import *

# 1. Read CSV
df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales2.csv",
    header=True,
    inferSchema=True
)

# 2. Filter records
filtered_df = df.filter(col("total_amount") > 1000)

# 3. Add ingestion date
final_df = filtered_df.withColumn(
    "ingestion_date",
    current_date()
)

# 4. Write result as Delta
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cyntexa_dev.sales.sales_table")

5. Read a JSON file with nested structure and flatten at least one nested field using dot notation or
explode().

In [0]:
%python
# Read JSON file
from pyspark.sql.functions import col
# Read JSON file
df = spark.read.json('/Volumes/cyntexa_dev/sales/raw/*.json')


# Flatten a nested field using dot notation
flattened_df = df.select(
    "customer_id",
    col("personal.first_name").alias("first_name"),
    col("personal.last_name").alias("last_name"),
    col("personal.gender").alias("gender"),
    col("contact.email").alias("email"),
    col("contact.phone").alias("phone"),
    col("contact.address.city").alias("city"),
    col("contact.address.state").alias("state"),
    col("contact.address.country").alias("country"),
)

flattened_df.show()

6. Call .explain() on a multi-step transformation chain and identify, from the physical plan, which steps
got pipelined together versus which required a shuffle.

In [0]:
%python
from pyspark.sql.functions import *

df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales2.csv",
    header=True,
    inferSchema=True
)

df = (
    df.filter(col("total_amount") > 1000)
.withColumn("tax" , col("total_amount") * 0.18)
.groupBy(col("customer_id"))
.agg(sum(col("total_amount")).alias("total_amount"))
)

df.explain()